# E-Commerce Analytics 360
## Étape 4 — Power BI Design Guide

> **Prérequis** : avoir complété les étapes 1, 2 et 3. Les fichiers CSV exportés depuis Colab doivent être disponibles sur ton PC.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 3h à 4h |

> 💡 **Ce notebook est un guide de conception.** Il ne contient pas de code Python. Tu vas travailler directement dans Power BI Desktop en suivant les instructions ci-dessous.

---

### Objectif
Transformer les données SQL en un dashboard décisionnel **5 pages** permettant à M. Diallo
de piloter : performance commerciale · produits · clients · funnel · satisfaction

---

## 1. Sources de données à importer

Les données ont été préparées dans les étapes précédentes et exportées en CSV depuis Colab. Tu as besoin des fichiers suivants sur ton PC :

## Tables à importer

| Table | Type | Rôle |
|---|---|---|
| `dim_customers` | Dimension | Profil clients |
| `dim_products` | Dimension | Catalogue produits |
| `fact_orders` | Fait | Commandes |
| `fact_order_items` | Fait | Lignes de commande |
| `fact_reviews` | Fait | Avis clients |
| `fact_web_logs` | Fait | Comportement web |
| `fact_ecommerce_analytics` | Analytique | Vue consolidée |
| `clients_rfm_segments` | Analytique | Segmentation RFM |

### Comment les récupérer depuis Colab

Dans Colab → panneau latéral gauche → icône **dossier** → clic droit sur le fichier → **Download**.

## Étapes détaillées

### Importer dans Power BI
1. Ouvre Power BI Desktop
2. **Obtenir les données** → **Texte/CSV**
3. Sélectionne chaque fichier
4. Choisis le mode **Import**
5. Vérifie les types de colonnes dans Power Query avant de charger



---

## Table Calendrier (à créer dans Power BI)

Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "MMMM", "fr-FR"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR"),
    "Est_Weekend",   IF(WEEKDAY([Date], 2) >= 6, 1, 0),
    "Annee_Mois",    FORMAT([Date], "YYYY-MM")
)

# 2. Modèle de données — Schéma en étoile

## Relations à configurer dans Power BI

```
dim_customers[customer_id] ──→ fact_orders[customer_id]
dim_customers[customer_id] ──→ fact_reviews[customer_id]
dim_customers[customer_id] ──→ fact_web_logs[customer_id]
dim_customers[customer_id] ──→ clients_rfm_segments[customer_id]
fact_orders[order_id]    ──→ fact_order_items[order_id]
dim_products[product_id] ──→ fact_order_items[product_id]
fact_orders[order_id]    ──→ fact_reviews[order_id]
```



---

# 3. Structure globale — 5 pages décisionnelles

| Page | Question business | Visuels clés |
|---|---|---|
| **1. Overview** | Comment évolue la performance globale ? | KPI cards, courbe CA, histogramme catégories |
| **2. Produits** | Quels produits/catégories génèrent le plus de valeur ? | Top 10, scatter CA vs marge |
| **3. Clients** | Qui sont nos meilleurs clients et segments ? | RFM, top clients, carte géo |
| **4. Funnel digital** | Où perd-on les utilisateurs ? | Funnel, taux conversion par device/source |
| **5. Satisfaction** | Quel est le niveau de satisfaction ? | Distribution ratings, évolution note |

## Principes de design
- 1 page = 1 question business principale
- KPIs en haut, tendances au centre, détails en bas
- Filtres globaux : Période · Pays · Canal · Catégorie · Segment client
- Palette cohérente : `#534AB7` violet · `#1D9E75` vert · `#EF9F27` orange · `#E24B4A` rouge

---

Crée une table `_Mesures` dans Power BI pour organiser toutes les mesures.

In [ ]:
// ── KPIs de base ─────────────────────────────────────────────────
CA Total =
CALCULATE(
    SUM(fact_order_items[line_revenue]),
    fact_orders[order_status] = "Livree"
)

Marge Totale =
CALCULATE(
    SUM(fact_order_items[line_margin]),
    fact_orders[order_status] = "Livree"
)

Taux de Marge =
DIVIDE([Marge Totale], [CA Total], 0)

Nb Commandes =
CALCULATE(
    DISTINCTCOUNT(fact_orders[order_id]),
    fact_orders[order_status] = "Livree"
)

Nb Clients =
DISTINCTCOUNT(dim_customers[customer_id])

Panier Moyen =
DIVIDE([CA Total], [Nb Commandes], 0)

Quantité Vendue =
CALCULATE(
    SUM(fact_order_items[quantite]),
    fact_orders[order_status] = "Livree"
)

Note Moyenne =
AVERAGE(fact_reviews[rating])

Nb Avis = COUNTROWS(fact_reviews)

Nb Produits Critiques = 
CALCULATE(
    DISTINCTCOUNT(fact_order_items[product_id]),
    FILTER(
        VALUES(fact_order_items[product_id]),
        CALCULATE(AVERAGE(fact_reviews[rating])) < 3
    )
)

In [ ]:
// ── Évolution temporelle (variation M/M) ──────────────────────────
Variation CA % = 
VAR _ca_actuel = [CA Total]
VAR _ca_precedent = CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_ca_actuel - _ca_precedent, _ca_precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)

Variation Marge % = 
VAR marge_actuelle = [Marge Totale]
VAR marge_precedente = CALCULATE([Marge Totale], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(marge_actuelle - marge_precedente, marge_precedente, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)


Variation commande = 
VAR nb_commandes_actuel = [Nb Commandes]
VAR nb_commandes_precedent = CALCULATE([Nb Commandes], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(nb_commandes_actuel - nb_commandes_precedent, nb_commandes_precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)

Variation client = 
VAR clients_actuel = [Nb clients]
VAR clients_precedente = CALCULATE([Nb clients], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(clients_actuel - clients_precedente, clients_precedente, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)

Variation panier = 
VAR panier_actuel = [Panier Moyen]
VAR panier_precedent = CALCULATE([Panier Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(panier_actuel - panier_precedent, panier_precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)

Variation note = 
VAR note_actuelle = [Note Moyenne]
VAR note_precedente = CALCULATE([Note Moyenne], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(note_actuelle - note_precedente, note_precedente, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(
    _pct > 0,
    UNICHAR(9650) & " " & _format,
    UNICHAR(9660) & " " & _format
)

-- Creer cette mseure pour colorer les deltas en fonction de leur signe
formatage_ca = 
VAR _ca_actuel = [CA Total]
VAR _ca_precedent = CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_ca_actuel - _ca_precedent, _ca_precedent, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)

formatage_marge =  
VAR marge_actuelle = [Marge Totale]
VAR marge_precedente = CALCULATE([Marge Totale], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(marge_actuelle - marge_precedente, marge_precedente, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)

formatage_commande =    
VAR nb_commandes_actuel = [Nb Commandes]
VAR nb_commandes_precedent = CALCULATE([Nb Commandes], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(nb_commandes_actuel - nb_commandes_precedent, nb_commandes_precedent, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)

formatage_client =          
VAR clients_actuel = [Nb clients]
VAR clients_precedente = CALCULATE([Nb clients], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(clients_actuel - clients_precedente, clients_precedente, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)

formatage_panier =  
VAR panier_actuel = [Panier Moyen]
VAR panier_precedent = CALCULATE([Panier Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(panier_actuel - panier_precedent, panier_precedent, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)   

formatage_note =    
VAR note_actuelle = [Note Moyenne]
VAR note_precedente = CALCULATE([Note Moyenne], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(note_actuelle - note_precedente, note_precedente, 0)
RETURN
SWITCH(
    TRUE(),
    _pct>0, "Green",
     _pct<0, "Red",
     "Grey"
)   


// ── YTD (Year-To-Date) ─────────────────────────────────────────────
CA YTD =
TOTALYTD([CA Total], dim_date[date])


Marge YTD =
TOTALYTD([Marge Totale], dim_date[date])

// ── Clients & RFM ────────────────────────────────────────────────

`CA Moyen par Client =
DIVIDE([CA Total], [Nb Clients], 0)`

`Nb Commandes par Client =
DIVIDE([Nb Commandes], [Nb Clients], 0)`

`% Clients VIP =
DIVIDE(
    CALCULATE(
        DISTINCTCOUNT(fact_ecommerce_analytics[customer_id]),
        fact_ecommerce_analytics[is_vip] = 1
    ),
    [Nb Clients],
    0
)`

`Nb Clients Champions =
CALCULATE(
    DISTINCTCOUNT(clients_rfm_segments[customer_id]),
    clients_rfm_segments[segment_rfm] = "Champions"
)`

`% Clients Actifs (RFM) =
DIVIDE(
    CALCULATE(
        COUNTROWS(clients_rfm_segments),
        clients_rfm_segments[segment_rfm] IN {"Champions","Fidèles","Nouveaux prometteurs"}
    ),
    COUNTROWS(clients_rfm_segments),
    0
)`

// ── Funnel digital ───────────────────────────────────────────────

`Nb Sessions =
DISTINCTCOUNT(fact_web_logs[session_id])`

`Nb Vues =
CALCULATE(
    COUNTROWS(fact_web_logs),
    SEARCH("fiche_produit", fact_web_logs[page], 1, 0) > 0
)`

`Nb Ajouts Panier =
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("panier", fact_web_logs[page], 1, 0) > 0
)`

`Nb Achats =
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("confirmation", fact_web_logs[page], 1, 0) > 0
)`

`Taux Conversion =
DIVIDE([Nb Achats], [Nb Sessions], 0)`

// ── Commandes à risque ───────────────────────────────────────────

`Nb Commandes Risque =
CALCULATE(
    DISTINCTCOUNT(fact_ecommerce_analytics[order_id]),
    fact_ecommerce_analytics[is_risky_order] = 1
)`

`Taux Commandes Risque =
DIVIDE([Nb Commandes Risque], [Nb Commandes], 0)`

`Délai Livraison Moyen =
AVERAGE(fact_orders[delivery_delay])`

`Taux Non Livré =
DIVIDE(
    CALCULATE(
        DISTINCTCOUNT(fact_orders[order_id]),
        fact_orders[order_status] <> "Livree"
    ),
    DISTINCTCOUNT(fact_orders[order_id]),
    0
)`

// ── Contribution canal ────────────────────────────────────────────

`Contribution Canal % =
DIVIDE(
    [CA Total],
    CALCULATE([CA Total], ALL(fact_orders[canal])),
    0
)`

In [ ]:
COLORS = {
    "Bleu":   "#4f8ef7",
    "Teal":   "#2dd4bf",
    "Amber":  "#f59e0b",
    "Violet": "#8b5cf6",
    "Vert":   "#22c55e",
    "rose":   "#f43f5e",
}


---

# Pages détaillées du dashboard

---

## Page 1 — Overview : "Comment évolue la performance globale ?"

### Layout
```
┌──────────────────────────────────────────────────────────────┐
│  KPI CARDS : CA Total · Marge · Nb Commandes · Clients · Panier moy. · Note moy.  │
├──────────────────────────────────────────────────────────────┤
│  Courbe : CA mensuel + Variation MoM (axe secondaire)        │
│  Histogramme : CA par catégorie (coloré par taux de marge)   │
├──────────────────────────────────────────────────────────────┤
│  Bar chart : CA par canal · Donut : répartition par statut   │
└──────────────────────────────────────────────────────────────┘
```

### Instructions construction
1. **KPI Cards** : utiliser le visuel "Carte" ou "Nouvelle carte"
   - Valeur : `[CA Total]`
   - Étiquette secondaire : `[Variation CA MoM %]` (vert/rouge selon signe)
2. **Courbe CA mensuel** : axe X = `Calendrier[Annee_Mois]`, Y = `[CA Total]`
3. **Histogramme catégories** : axe X = `dim_products[categorie]`, Y = `[CA Total]`
   - Formatage conditionnel : couleur de barre selon `[Taux de Marge]`

---

## Page 2 — Produits : "Quels produits génèrent le plus de valeur ?"

### Layout
```
┌──────────────────────────────────────────────────────────────┐
│  KPI CARDS : Quantité vendue · CA produits · Marge produits  │
├──────────────────────────────────────────────────────────────┤
│  Top 10 produits par CA (barres horizontales)                │
│  Top catégories par CA et marge (barres groupées)            │
├──────────────────────────────────────────────────────────────┤
│  Scatter : CA (X) vs Marge (Y) par produit — taille = qté   │
│  Table : Produit · Catégorie · Qté · CA · Marge              │
└──────────────────────────────────────────────────────────────┘
```

### Formatage conditionnel table produits
- Colonne `Marge` : fond rouge si taux marge < 20%, vert si > 35%
- Colonne `Qté` : barre de données intégrée

---

## Page 3 — Clients : "Qui génère le plus de valeur ?"

### Layout
```
┌──────────────────────────────────────────────────────────────┐
│  KPI CARDS : Nb clients · CA/client · Cmdes/client · % VIP  │
├──────────────────────────────────────────────────────────────┤
│  Donut : Répartition segments RFM                            │
│  Bar chart : CA par segment RFM (coloré Champions=vert)      │
├──────────────────────────────────────────────────────────────┤
│  Map / Bar chart : CA par pays                               │
│  Table : Top 15 clients (customer_id, segment, CA, nb cmde)  │
└──────────────────────────────────────────────────────────────┘
```

### Visuel Segments RFM — DAX couleur conditionnelle

// Couleur segment — utiliser dans formatage conditionnel

Couleur Segment =
SWITCH(
    SELECTEDVALUE(clients_rfm_segments[segment_rfm]),
    "Champions",                   "#1D9E75",
    "Fidèles",                     "#534AB7",
    "Gros dépensiers occasionnels","#EF9F27",
    "Nouveaux prometteurs",        "#5DCAA5",
    "À réactiver",                 "#E24B4A",
    "Dormants",                    "#888780"
)

---

## Page 4 — Funnel digital : "Où perd-on les utilisateurs ?"

### Layout
```
┌──────────────────────────────────────────────────────────────┐
│  KPI CARDS : Nb Sessions · Nb Vues · Ajouts panier · Achats  │
│  KPI CARD : Taux de conversion global                        │
├──────────────────────────────────────────────────────────────┤
│  Funnel chart : Sessions → Vues produit → Panier → Achat     │
│  Bar chart : Taux de conversion par canal d'acquisition      │
├──────────────────────────────────────────────────────────────┤
│  Bar chart : Conversion par device (Mobile vs Desktop)       │
│  Table : Canal · Vues · Panier · Achats · Taux conv.         │
└──────────────────────────────────────────────────────────────┘
```

### Visuel Funnel
Utiliser le visuel natif "Entonnoir" de Power BI :
- Catégorie : étapes (Vues, Panier, Confirmation)
- Valeurs : `[Nb Vues]`, `[Nb Ajouts Panier]`, `[Nb Achats]`

---

## Page 5 — Satisfaction : "Quel est le niveau de satisfaction ?"

### Layout
```
┌──────────────────────────────────────────────────────────────┐
│  KPI CARDS : Note moy. · Nb avis · % avis négatifs · Nb prod. < 3 │
├──────────────────────────────────────────────────────────────┤
│  Histogramme : distribution des ratings (1→5)                │
│  Bar chart : Top 10 produits avec les plus mauvaises notes   │
├──────────────────────────────────────────────────────────────┤
│  Courbe : évolution mensuelle de la note moyenne             │
│  Table : Produit · Catégorie · Note moy · Nb avis            │
└──────────────────────────────────────────────────────────────┘
```


---

# 11. Filtres & slicers recommandés

## Filtres globaux (toutes les pages)
| Filtre | Source | Type |
|---|---|---|
| Période | `dim_date[date]` | Date range |
| Année | `Calendrier[Annee]` | Liste déroulante |
| Trimestre | `Calendrier[Trimestre]` | Boutons |

## Filtres par page
| Page | Filtres spécifiques |
|---|---|
| Overview | Canal, Statut commande |
| Produits | Catégorie, Marque, Fourchette prix |
| Clients | Segment RFM, Pays, Ville |
| Funnel | Device, Source (canal), Page |
| Satisfaction | Catégorie, Fourchette rating |

## Conseil UX
- Placer les filtres globaux dans un volet latéral gauche ou dans une page de filtres dédiée
- Utiliser des `Slicers` à boutons pour les listes courtes (< 6 valeurs)
- Utiliser des `Slicers` liste déroulante pour les listes longues (pays, villes)

---

# 12. Bonnes pratiques de design Power BI

## ✅ À faire
- Titres de pages = questions business (pas "Page 1")
- Afficher les unités : k€, %, nombre
- Hiérarchie visuelle : KPI en haut · tendances au centre · détails en bas
- Couleurs sémantiques : vert = bon, rouge = alerte, orange = vigilance
- Formatage conditionnel sur toutes les colonnes numériques dans les tables
- Ajouter une info-bulle (tooltip) sur chaque KPI card avec la définition
- Tester le dashboard avec les filtres croisés désactivés/activés

## ❌ À éviter
- Plus de 5-6 visuels par page
- Plusieurs palettes de couleurs différentes
- Tables avec > 6 colonnes visibles
- Visuels sans titre
- KPI sans contexte (toujours indiquer la période et la cible)
- Décimales inutiles (écrire 1 245 000 € et non 1 244 987,43 €)

---

# 13. Storytelling — Ordre de présentation

## Séquence narrative pour la restitution au comité de direction

1. **Performance globale** (page Overview)
   → *"Voici où on en est : CA, marge, commandes"*

2. **Moteurs de performance** (page Produits)
   → *"Ce qui génère notre CA et notre marge"*

3. **Valeur client** (page Clients)
   → *"Qui sont nos meilleurs clients et comment les segmenter"*

4. **Conversion digitale** (page Funnel)
   → *"Où on perd les utilisateurs et ce qu'il faut corriger"*

5. **Qualité perçue** (page Satisfaction)
   → *"Ce que nos clients pensent de nous"*

6. **Actions prioritaires** (page Insights)
   → *"Ce que nous devons faire maintenant"*

## Message final

> L'apprenant ne doit pas seulement savoir créer un dashboard.
> Il doit être capable de répondre à la question :
>
> **"Que doit faire ShopAfrica+ dans les 3 prochains mois ?"**

---

## Mini checklist ✅
- [x] Connexion Power BI → SQL Server documentée (+ fallback CSV)
- [x] Modèle en étoile + relations définis
- [x] Table Calendrier créée en DAX
- [x] 32 mesures DAX complètes et commentées
- [x] Architecture 6 pages avec layouts détaillés
- [x] Filtres globaux + filtres par page
- [x] Bonnes pratiques de design
- [x] Storytelling exécutif structuré
- [x] Score de santé global (mesure composite)

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.